# 文档加载器 Document Loaders

## 1.记载txt



In [ ]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader(
    file_path = "../asset/load/01-langchain-utf-8.txt",
    encoding = "utf-8"
)

docs = loader.load()

print(docs)

Document对象有两个重要的属性：
1）page_content: 真正的文档内容，字符串类型。
2）metadata: 文档内容的元数据，字典类型

In [ ]:
print(docs[0])
print(type(docs[0]))

print(docs[0].metadata)
print(docs[0].page_content)

In [ ]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader(
    file_path = "../asset/load/01-langchain-gbk.txt",
    encoding = "gbk",   # 务必编码和解码一致
)

docs = loader.load()

print(docs)

## 2.加载CSV格式

In [ ]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(
    file_path = "../asset/load/02-load.csv"
)

docs = loader.load()

print(docs)

## 3.加载JSON

LangChain提供的JSON格式的文档加载器是`JSONLoader`。在实际应用场景中，JSON格式的数据占有很大比例，而且JSON的形式也是多样的。我们需要特别关注。

JSONLoader 使用指定的 `jq结构`来解析 JSON 文件。jq是一个轻量级的命令行 JSON 处理器 ，可以对 JSON 格式的数据进行各种复杂的处理，包括数据过滤、映射、减少和转换，是处理 JSON 数据的`首选工具之一`。

常见 jq schema 参考：

JSON        -> [{"text": ...}, {"text": ...}, {"text": ...}]
jq_schema   -> ".[].text"

JSON        -> {"key": [{"text": ...}, {"text": ...}, {"text": ...}]}
jq_schema   -> ".key[].text"

JSON        -> ["...", "...", "..."]
jq_schema   -> ".[]"


举例1：使用JSONLoader文档器加载

In [ ]:
from langchain_community.document_loaders import JSONLoader
from rich import print as rprint

# 情况1
# json_loader = JSONLoader(
#     file_path = "../asset/load/03-load.json",
#     jq_schema = ".",   # 直接提取完整的JSON对象（包括所有字段）
#     text_content=False    # 保持原始JSON结构，将提取的数据转换成JSON字符串存入page_content字段中
# )

# 情况2
# .messages[].content：遍历.messages[]中所有元素，从每一个元素中提取.content字段
json_loader = JSONLoader(
    file_path = "../asset/load/03-load.json",
    jq_schema=".messages[].content"
)


# 加载
docs = json_loader.load()
rprint(docs)

举例2：提取03-response.json文件中指定的文本

In [21]:
from langchain_community.document_loaders import JSONLoader
from rich import print as rprint

file_path = "../asset/load/03-response.json"

# 需求1：提取data.items
# loader = JSONLoader(
#     file_path = file_path,
#     jq_schema = ".data.items[]",
#     text_content = False   # 提取内容是否为字符串格式，默认是True
# )

# 需求2：提取data.items[].content中的数据
# loader = JSONLoader(
#     file_path = file_path,
#     jq_schema = ".data.items[].content",
# )

# 需求3：提取data.items中指定字段的数据
loader = JSONLoader(
    file_path = file_path,
    jq_schema = """
        .data.items[] | {
            author,
            created_at,
            content: (.title + "\n" + .content)
        }
    """,
    text_content = False,    # 提取内容是否为字符串格式
)

data = loader.load()
rprint(data)

[
    Document(
        metadata={
            'source': 'C:\\Users\\如玧\\Desktop\\langchain_learn\\asset\\load\\03-response.json',
            'seq_num': 1
        },
        page_content='{"author": {"id": "user_1", "name": "Alice"}, "created_at": "2023-10-05T08:12:33Z", 
"content": "Understanding JSONLoader\\nThis article explains how to parse API responses..."}'
    ),
    Document(
        metadata={
            'source': 'C:\\Users\\如玧\\Desktop\\langchain_learn\\asset\\load\\03-response.json',
            'seq_num': 2
        },
        page_content='{"author": {"id": "user_2", "name": "Bob"}, "created_at": "2023-10-05T09:15:21Z", "content": 
"Advanced jq Schema Patterns\\nLearn to handle nested structures with..."}'
    ),
    Document(
        metadata={
            'source': 'C:\\Users\\如玧\\Desktop\\langchain_learn\\asset\\load\\03-response.json',
            'seq_num': 3
        },
        page_content='{"author": {"id": "user_3", "name": "Charlie"}, "created_at": "2023-10-05T10:03:47Z", 
"content": "LangChain Metadata Handling\\nBest practices for preserving metadata..."}'
    )
]